# Day 6 — Policy Interpretation and Conclusions

**Project:** The Impact of Inflation and Housing Costs on Social Inequality and Future Trends in the UK  
**Day:** 6 of 7 — Policy Interpretation and Conclusions  

---

## Purpose

This notebook translates the quantitative outputs from Days 3–5 into economic conclusions:
- Runs a +1 percentage point Bank Rate shock scenario using the trained Day 4 models
- Maps the distributional impact across income deciles
- Evaluates two policy responses (targeted rent subsidy vs rent controls)
- Drafts conclusions for the Day 7 final report

**This notebook is primarily interpretive.** It does not retrain models or reopen the holdout for tuning.

---

### Day 1 Hypotheses (for reference)
- **H1:** Higher inflation reduces real income more for lower-income deciles
- **H2:** Higher housing costs increase the housing expenditure share more for lower-income deciles
- **H3:** CPI/CPIH and Bank Rate have explanatory power for short-term housing market pressure

---
## 00 — Setup

In [1]:
import sys
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # headless for nbconvert
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

# Suppress minor pandas/sklearn warnings that are not relevant to this analysis.
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:.3f}'.format)
np.random.seed(42)

# ── Paths ────────────────────────────────────────────────────────────────────
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
# Add the project root to Python's module search path so that
# "from src.day6_interest_scenario import ..." resolves correctly.
sys.path.append(str(PROJECT_ROOT))

OUTPUTS    = PROJECT_ROOT / 'outputs'
FIGURES    = PROJECT_ROOT / 'figures' / 'day6'
DATA_PROC  = PROJECT_ROOT / 'data' / 'processed'
MODELS_DIR = PROJECT_ROOT / 'models'

OUTPUTS.mkdir(exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print('Project root :', PROJECT_ROOT)
print('Outputs dir  :', OUTPUTS)
print('Figures dir  :', FIGURES)

Project root : /Users/hongmiaozhu/PycharmProjects/Project 
Outputs dir  : /Users/hongmiaozhu/PycharmProjects/Project /outputs
Figures dir  : /Users/hongmiaozhu/PycharmProjects/Project /figures/day6


In [2]:
# Try to import helper functions from the src/ folder.
# If the import fails (e.g. running from a different working directory), we fall back to
# defining the same functions inline below so the notebook still runs.
# ── Try importing reusable modules (fall back to inline if unavailable) ───────
try:
    from src.day6_interest_scenario import (
        load_models, load_model_ready_data,
        build_scenario, score_scenario,
        build_comparison_table, check_extrapolation,
        get_linear_regression_step,
    )
    from src.day6_policy_analysis import (
        load_income_decile_data, build_distributional_summary,
        build_policy_matrix, get_primary_recommendation,
        compute_mortgage_balance_from_data,
        FALLBACK_EXTRA_ANNUAL_COST,
    )
    print('✓ src modules imported successfully')
    MODULES_AVAILABLE = True
except ImportError as e:
    print(f'⚠  src import failed ({e}). Falling back to inline definitions.')
    MODULES_AVAILABLE = False

✓ src modules imported successfully


### Inline fallbacks (only executed if src import failed)

In [3]:
if not MODULES_AVAILABLE:
    FEATURE_COLS = [
        'inflation_rate_lag1', 'unemployment_rate_lag1',
        'bank_rate_lag1', 'bank_rate_lag3',
    ]
    # Fallback cost estimate used when the raw HPI file is unavailable.
    FALLBACK_EXTRA_ANNUAL_COST = 2_010.0  # approx £201k mortgage × 1pp

    def load_models(root):
        with open(root / 'models' / 'day4_linear_regression.pkl', 'rb') as f:
            lr = pickle.load(f)
        with open(root / 'models' / 'day4_random_forest.pkl', 'rb') as f:
            rf = pickle.load(f)
        return lr, rf

    def get_linear_regression_step(model):
        if hasattr(model, 'named_steps') and 'model' in model.named_steps:
            return model.named_steps['model']
        return model

    def load_model_ready_data(root):
        return pd.read_csv(root / 'data' / 'processed' / 'model_ready_day4.csv',
                           parse_dates=['year_month'])

    def build_scenario(df, shock_lag1=1.0, shock_lag3=1.0):
        s = df.copy()
        s['bank_rate_lag1'] += shock_lag1
        s['bank_rate_lag3'] += shock_lag3
        return s

    def score_scenario(model, base, scen, name):
        bp = model.predict(base[FEATURE_COLS].values)
        sp = model.predict(scen[FEATURE_COLS].values)
        return pd.DataFrame({'year_month': base['year_month'].values,
                              'model': name,
                              'baseline_predicted': bp,
                              'scenario_predicted': sp,
                              'delta': sp - bp})

    def build_comparison_table(lr, rf, df, shock_lag1=1.0, shock_lag3=1.0):
        scen = build_scenario(df, shock_lag1, shock_lag3)
        rows = []
        for mdl, name in [(lr, 'linear_regression'), (rf, 'random_forest')]:
            sc = score_scenario(mdl, df, scen, name)
            mb = sc['baseline_predicted'].mean()
            ms = sc['scenario_predicted'].mean()
            md = sc['delta'].mean()
            rows.append({'model': name,
                         'mean_baseline_predicted': round(mb, 3),
                         'mean_scenario_predicted': round(ms, 3),
                         'mean_delta': round(md, 3),
                         'pct_change': round(md / mb * 100, 3)})
        return pd.DataFrame(rows)

    def check_extrapolation(df, shock=1.0):
        return {'obs_min_lag1': df['bank_rate_lag1'].min(),
                'obs_max_lag1': df['bank_rate_lag1'].max(),
                'obs_min_lag3': df['bank_rate_lag3'].min(),
                'obs_max_lag3': df['bank_rate_lag3'].max(),
                'scenario_max_lag1': df['bank_rate_lag1'].max() + shock,
                'scenario_max_lag3': df['bank_rate_lag3'].max() + shock,
                'extrapolates': True}

    def load_income_decile_data(root, year_month='2024-03'):
        df = pd.read_csv(root / 'data' / 'processed' / 'day2_income_decile_clean.csv')
        return df[df['year_month'] == year_month].sort_values('decile').reset_index(drop=True)

    def build_distributional_summary(income_df, extra_annual_cost=FALLBACK_EXTRA_ANNUAL_COST):
        df = income_df[['decile', 'income_value']].copy()
        df['estimated_extra_annual_cost'] = extra_annual_cost
        df['extra_cost_as_pct_of_income'] = (extra_annual_cost / df['income_value'] * 100).round(2)
        return df.reset_index(drop=True)

    def build_policy_matrix():
        return pd.DataFrame([
            {'policy_name': 'Targeted Rent Subsidy',
             'objective': 'Reduce housing cost burden for lowest-income renters',
             'target_group': 'Renters in income deciles 1–3',
             'expected_benefit': 'Direct income relief; preserves market price signals',
             'main_risk': 'May push rents up if supply is inelastic (landlord incidence)',
             'fiscal_cost_indication': 'Medium',
             'economic_theory_basis': 'Demand-side transfer; incidence depends on supply elasticity',
             'primary_recommendation': True},
            {'policy_name': 'Rent Controls',
             'objective': 'Cap rent growth to improve near-term affordability for all renters',
             'target_group': 'All private renters',
             'expected_benefit': 'Immediate reduction in rents; broad coverage',
             'main_risk': 'Reduced supply, maintenance decline, potential black market',
             'fiscal_cost_indication': 'Low',
             'economic_theory_basis': 'Price ceiling below equilibrium creates shortage',
             'primary_recommendation': False},
        ])

    def get_primary_recommendation():
        return {'policy_name': 'Targeted Rent Subsidy',
                'recommendation_justification':
                    'Preserves market signals while targeting most-exposed households.'}

    def compute_mortgage_balance_from_data(project_root, ltv_ratio=0.75):
        # Stub fallback — returns the static fallback values when the src module
        # is unavailable. The real function in src/day6_policy_analysis.py reads
        # from the raw UK HPI file downloaded by main.py.
        mortgage_balance = round(FALLBACK_AVERAGE_PRICE * ltv_ratio, 0)
        extra_cost = round(mortgage_balance * 0.01, 0)
        return {
            'latest_month': 'unavailable',
            'average_house_price': FALLBACK_AVERAGE_PRICE,
            'ltv_ratio': ltv_ratio,
            'estimated_mortgage_balance': mortgage_balance,
            'extra_annual_cost': extra_cost,
            'source_file': 'fallback',
        }

    FALLBACK_AVERAGE_PRICE = 268_000  # approximate UK average house price, Jan 2026

    print('Inline fallbacks defined.')

---
## 01 — Load Day 3–5 Outputs

We load every file produced in Days 3–5. No model retraining occurs here.

In [4]:
# ── Day 5 official metrics ────────────────────────────────────────────────────
metrics_df = pd.read_csv(OUTPUTS / 'day5_official_holdout_metrics.csv')
print('=== Day 5 Official Holdout Metrics ===')
display(metrics_df[['model', 'MAE', 'RMSE', 'R2']])

# ── Holdout predictions ───────────────────────────────────────────────────────
holdout_df = pd.read_csv(OUTPUTS / 'day4_model_comparison_holdout_predictions.csv',
                          parse_dates=['year_month'])
print(f'\nHoldout predictions: {holdout_df.shape[0]} rows')
print(f'  Date range: {holdout_df["year_month"].min().date()} → {holdout_df["year_month"].max().date()}')

# ── Residuals ─────────────────────────────────────────────────────────────────
residuals_df = pd.read_csv(OUTPUTS / 'day5_residuals.csv', parse_dates=['year_month'])
print(f'\nResiduals: {residuals_df.shape[0]} rows')

# ── Monthly macro ─────────────────────────────────────────────────────────────
macro_df = pd.read_csv(DATA_PROC / 'day2_merged_monthly_macro.csv',
                        parse_dates=['year_month'])
print(f'\nMonthly macro: {macro_df.shape}')

# ── Income by decile ──────────────────────────────────────────────────────────
income_all_df = pd.read_csv(DATA_PROC / 'day2_income_decile_clean.csv')
print(f'\nIncome by decile: {income_all_df.shape}')
print(f'  Years available: {sorted(income_all_df["year_month"].unique())}')

# ── BBC sentiment ─────────────────────────────────────────────────────────────
bbc_df = pd.read_csv(DATA_PROC / 'bbc_sentiment.csv')
print(f'\nBBC headlines: {bbc_df.shape[0]} rows')
print(f'  Sentiment distribution:\n{bbc_df["vader_label"].value_counts().to_string()}')

=== Day 5 Official Holdout Metrics ===


,model,MAE,RMSE,R2
0,linear_regression,6.377,6.783,-28.506
1,random_forest,3.371,3.616,-7.386



Holdout predictions: 12 rows
  Date range: 2025-02-01 → 2026-01-01

Residuals: 24 rows

Monthly macro: (59, 7)

Income by decile: (50, 7)
  Years available: ['2020-03', '2021-03', '2022-03', '2023-03', '2024-03']

BBC headlines: 50 rows
  Sentiment distribution:
vader_label
negative    23
neutral     14
positive    13


> **Note:** All Day 3–5 outputs loaded. This notebook interprets them — it does not retrain models or reopen the holdout.

Key observations from the metrics:
- Both models produce **negative R²** on the holdout, indicating that on unseen data neither model outperforms a naïve mean prediction in terms of variance explained.
- The random forest achieves lower MAE (3.37 vs 6.38) and RMSE (3.62 vs 6.78), suggesting better point accuracy even if the variance explanation is limited.
- These results inform our caution: the scenario analysis below yields model-implied associations, not reliable forecasts.

---
## 02 — Build the +1pp Bank Rate Scenario

**Scenario design:** A sustained +1 percentage point increase in Bank Rate. Because a sustained rate rise would filter into both the one-month and three-month lags, we shock **both** `bank_rate_lag1` and `bank_rate_lag3` by +1.0 pp. All other features are held at their observed values.

This is a **counterfactual**, not a forecast. We are asking: *given the model and the observed feature values, what would predictions look like if Bank Rate had been 1 pp higher at every point in the sample?*

In [5]:
# ── Load models and model-ready data ─────────────────────────────────────────
lr_model, rf_model = load_models(PROJECT_ROOT)
linear_step = get_linear_regression_step(lr_model)
model_df = load_model_ready_data(PROJECT_ROOT)

print(f'Model-ready dataset: {model_df.shape[0]} rows')
print(f'  Date range: {model_df["year_month"].min().date()} → {model_df["year_month"].max().date()}')
print(f'  Linear regression — intercept: {linear_step.intercept_:.3f}, coefs: {linear_step.coef_}')

print("\n  Coefficient breakdown:")
feature_names = ["inflation_rate_lag1", "unemployment_rate_lag1", "bank_rate_lag1", "bank_rate_lag3"]
for name, coef in zip(feature_names, linear_step.coef_):
    print(f"    {name:30s} → {coef:+.4f}")
net_bank_rate_effect = linear_step.coef_[2] + linear_step.coef_[3]
print(f"\n  Net linear effect of +1pp on BOTH bank_rate_lag1 and bank_rate_lag3:")
print(f"    = ({linear_step.coef_[2]:+.4f}) + ({linear_step.coef_[3]:+.4f}) = {net_bank_rate_effect:+.4f} index points")
print(f'  Random forest — n_estimators: {rf_model.n_estimators}')

Model-ready dataset: 56 rows
  Date range: 2021-06-01 → 2026-01-01
  Linear regression — intercept: 109.225, coefs: [ 0.4082511  -3.96141955 -0.14988756  1.22415015]

  Coefficient breakdown:
    inflation_rate_lag1            → +0.4083
    unemployment_rate_lag1         → -3.9614
    bank_rate_lag1                 → -0.1499
    bank_rate_lag3                 → +1.2242

  Net linear effect of +1pp on BOTH bank_rate_lag1 and bank_rate_lag3:
    = (-0.1499) + (+1.2242) = +1.0743 index points
  Random forest — n_estimators: 200


In [6]:
# ── Extrapolation check ───────────────────────────────────────────────────────
extrap = check_extrapolation(model_df, shock=1.0)

print('=== Extrapolation Check: +1pp Bank Rate Shock ===')
print(f'  bank_rate_lag1 observed range : [{extrap["obs_min_lag1"]:.2f}, {extrap["obs_max_lag1"]:.2f}] pp')
print(f'  bank_rate_lag1 after shock     : max = {extrap["scenario_max_lag1"]:.2f} pp')
print(f'  bank_rate_lag3 observed range : [{extrap["obs_min_lag3"]:.2f}, {extrap["obs_max_lag3"]:.2f}] pp')
print(f'  bank_rate_lag3 after shock     : max = {extrap["scenario_max_lag3"]:.2f} pp')
print()
if extrap['extrapolates']:
    print('⚠  EXTRAPOLATION FLAG: The +1pp shock pushes the Bank Rate features')
    print('   beyond the maximum observed in training. Results for the upper tail')
    print('   of the distribution should be treated with additional caution.')
else:
    print('✓ Scenario stays within observed training range.')

=== Extrapolation Check: +1pp Bank Rate Shock ===
  bank_rate_lag1 observed range : [0.10, 5.25] pp
  bank_rate_lag1 after shock     : max = 6.25 pp
  bank_rate_lag3 observed range : [0.10, 5.25] pp
  bank_rate_lag3 after shock     : max = 6.25 pp

⚠  EXTRAPOLATION FLAG: The +1pp shock pushes the Bank Rate features
   beyond the maximum observed in training. Results for the upper tail
   of the distribution should be treated with additional caution.


> **Robustness note:** The observed Bank Rate in the training data ranges from 0.10 pp to 5.25 pp. A +1pp shock applied to observations already at 5.25 pp would push the scenario to 6.25 pp, which lies **outside the observed training range**. For those observations the model is extrapolating. The linear model's prediction will still be mechanically well-defined (it extrapolates by construction), but the random forest may be less reliable at the boundary of its training distribution. **This does not invalidate the scenario — it simply means the predicted effect for the highest-rate rows should be read as indicative rather than precise.**

In [7]:
# ── Run the scenario ──────────────────────────────────────────────────────────
# shock_lag1 and shock_lag3 add +1 percentage point to both Bank Rate lag features,
# simulating a sustained rate rise. This is a counterfactual illustration, not a forecast.
scenario_df = build_scenario(model_df, shock_lag1=1.0, shock_lag3=1.0)

lr_scores = score_scenario(lr_model, model_df, scenario_df, 'linear_regression')
rf_scores = score_scenario(rf_model, model_df, scenario_df, 'random_forest')

# Row-level summary
print('=== Linear Regression: per-row delta summary ===')
print(lr_scores['delta'].describe().to_string())
print()
print('=== Random Forest: per-row delta summary ===')
print(rf_scores['delta'].describe().to_string())

=== Linear Regression: per-row delta summary ===
count   56.000
mean     1.074
std      0.000
min      1.074
25%      1.074
50%      1.074
75%      1.074
max      1.074

=== Random Forest: per-row delta summary ===
count   56.000
mean     1.144
std      3.040
min     -1.199
25%     -0.719
50%      0.000
75%      0.342
max     10.765


In [8]:
# ── Comparison table ──────────────────────────────────────────────────────────
comparison_table = build_comparison_table(lr_model, rf_model, model_df,
                                           shock_lag1=1.0, shock_lag3=1.0)

print('=== +1pp Bank Rate Scenario: Model Comparison ===')
display(comparison_table)

# Save
comparison_table.to_csv(OUTPUTS / 'day6_interest_scenario.csv', index=False)
print('\n✓ Saved: outputs/day6_interest_scenario.csv')

=== +1pp Bank Rate Scenario: Model Comparison ===


,model,mean_baseline_predicted,mean_scenario_predicted,mean_delta,pct_change
0,linear_regression,97.880,98.954,1.074,1.098
1,random_forest,98.527,99.671,1.144,1.161



✓ Saved: outputs/day6_interest_scenario.csv


In [9]:
# ── Visualise delta distribution for both models ──────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, scores, colour, label in [
    (axes[0], lr_scores, '#2166ac', 'Linear Regression'),
    (axes[1], rf_scores, '#d6604d', 'Random Forest'),
]:
    ax.plot(scores['year_month'], scores['delta'], color=colour, linewidth=1.5)
    ax.axhline(0, color='grey', linewidth=0.8, linestyle='--')
    ax.axhline(scores['delta'].mean(), color=colour, linewidth=1.0,
               linestyle=':', label=f'Mean delta = {scores["delta"].mean():.2f}')
    ax.set_title(f'{label}\nDelta: Scenario − Baseline', fontsize=11)
    ax.set_xlabel('Date')
    ax.set_ylabel('Δ House Price Index (index points)')
    ax.legend(fontsize=9)
    ax.tick_params(axis='x', rotation=30)

fig.suptitle('+1pp Bank Rate Shock: Model-Implied HPI Delta (Scenario − Baseline)',
             fontsize=13, y=1.02)
plt.tight_layout()
fig.savefig(FIGURES / 'figure_01_rate_scenario_delta.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: figures/day6/figure_01_rate_scenario_delta.png')

✓ Saved: figures/day6/figure_01_rate_scenario_delta.png


---
## 03 — Score and Interpret the Scenario

We now display the scenario table and provide an economic interpretation.

In [10]:
display(comparison_table)

lr_row = comparison_table[comparison_table['model'] == 'linear_regression'].iloc[0]
rf_row = comparison_table[comparison_table['model'] == 'random_forest'].iloc[0]

print(f"""
Linear Regression:
  Mean baseline predicted HPI : {lr_row['mean_baseline_predicted']:.2f}
  Mean scenario predicted HPI : {lr_row['mean_scenario_predicted']:.2f}
  Mean delta                  : {lr_row['mean_delta']:.2f} index points
  Percentage change           : {lr_row['pct_change']:.2f}%

Random Forest:
  Mean baseline predicted HPI : {rf_row['mean_baseline_predicted']:.2f}
  Mean scenario predicted HPI : {rf_row['mean_scenario_predicted']:.2f}
  Mean delta                  : {rf_row['mean_delta']:.2f} index points
  Percentage change           : {rf_row['pct_change']:.2f}%
""")

,model,mean_baseline_predicted,mean_scenario_predicted,mean_delta,pct_change
0,linear_regression,97.880,98.954,1.074,1.098
1,random_forest,98.527,99.671,1.144,1.161



Linear Regression:
  Mean baseline predicted HPI : 97.88
  Mean scenario predicted HPI : 98.95
  Mean delta                  : 1.07 index points
  Percentage change           : 1.10%

Random Forest:
  Mean baseline predicted HPI : 98.53
  Mean scenario predicted HPI : 99.67
  Mean delta                  : 1.14 index points
  Percentage change           : 1.16%



### Interpretation

The plotted delta is `scenario_predicted - baseline_predicted`. Positive values therefore mean the fitted model gives a higher HPI prediction after adding +1pp to both `bank_rate_lag1` and `bank_rate_lag3`; negative values would mean the opposite. In the current output, both models produce a small positive mean delta.

This should be interpreted carefully. It does **not** prove that higher Bank Rate causes higher house prices. It means that, within this short 2021–2026 model window and this specific lag structure, the shocked Bank Rate features are associated with slightly higher fitted HPI predictions. This can coexist with the broader economic expectation that higher rates often cool housing demand because the model is capturing correlations from one unusual macro period, not an identified causal mechanism.

The linear-regression line is nearly flat because the same +1pp shock is applied to the same lagged features in every row, and the model uses one fixed coefficient structure. The random forest can vary more across months because it allows nonlinear responses depending on the surrounding macro conditions.

**Critical caveat:** These are **model-implied associations**, not causal estimates or reliable forecasts. Day 5 showed negative holdout R² for both models, so the main value here is disciplined scenario interpretation rather than predictive confidence.

---
## 04 — Distributional Mapping

Here we connect the rate scenario to the inequality findings from Day 3. The data-derived extra annual cost is the same in absolute terms for all income groups — but it represents a very different share of disposable income across deciles.

**Data-derived calculation:** The mortgage balance is computed dynamically from the project's own UK HPI data — the same file that `main.py` downloads from the GOV.UK API. The latest available month's UK average house price is used, multiplied by a 75% loan-to-value ratio (25% deposit) to estimate the mortgage balance. A +1pp rate rise on an interest-only basis gives the extra annual cost shown above.

**Important assumption:** The same annual mortgage-cost increase benchmark is applied across all deciles. This is a stylised distributional illustration designed to show regressivity; it is **not** an estimate that every household faces exactly this payment increase. Actual impact depends on mortgage type, remaining term, fixed vs variable rate, debt size, whether the household rents or owns, and landlord pass-through.

When the pipeline is re-run with updated data, this calculation will reflect the latest available house prices automatically.

In [11]:
# ── Load 2024-03 income data ──────────────────────────────────────────────────
income_2024 = load_income_decile_data(PROJECT_ROOT, year_month='2024-03')
print('=== Income by Decile (2024-03) ===')
display(income_2024[['decile', 'income_value']])

=== Income by Decile (2024-03) ===


,decile,income_value
0,1,10725.000
1,2,19647.000
2,3,25009.000
3,4,29627.000
4,5,34244.000
5,6,39270.000
6,7,44887.000
7,8,52196.000
8,9,63025.000
9,10,108205.000


In [12]:
# Compute the mortgage balance from the project's own UK HPI data.
# This reads the latest available month from the same file that main.py
# downloads from the GOV.UK API, so the result updates automatically
# when the pipeline is re-run with newer data.
try:
    # ltv_ratio=0.75 means the borrower took out a mortgage for 75% of the house price.
    # The function multiplies that balance by 0.01 (1pp rate shock) to get the extra annual cost.
    mortgage_info = compute_mortgage_balance_from_data(PROJECT_ROOT, ltv_ratio=0.75)
    computed_extra_cost = mortgage_info['extra_annual_cost']

    print('=== Mortgage Balance — Computed From Project Data ===')
    print(f"  Source file: {mortgage_info['source_file']}")
    print(f"  Latest available month: {mortgage_info['latest_month']}")
    print(f"  UK average house price: £{mortgage_info['average_house_price']:,.0f}")
    print(f"  LTV ratio: {mortgage_info['ltv_ratio']:.0%} (25% deposit)")
    print(f"  Estimated mortgage balance: £{mortgage_info['estimated_mortgage_balance']:,.0f}")
    print(f"  Extra annual cost from +1pp shock: £{computed_extra_cost:,.0f}")
    print(f'\n  Note: this value updates automatically when main.py refreshes the UK HPI data.')

except FileNotFoundError as e:
    print(f'⚠ {e}')
    print(f'  Using fallback: £{FALLBACK_EXTRA_ANNUAL_COST:,.0f}/year')
    computed_extra_cost = FALLBACK_EXTRA_ANNUAL_COST

# Build the distributional impact table using the computed cost.
dist_df = build_distributional_summary(income_2024, extra_annual_cost=computed_extra_cost)

print(f'\n=== Distributional Impact of +1pp Rate Shock ===')
print(f'(Mortgage balance computed from latest UK HPI data → £{computed_extra_cost:,.0f}/year extra cost)')
print()
display(dist_df)

dist_df.to_csv(OUTPUTS / 'day6_distributional_summary.csv', index=False)
print('\n✓ Saved: outputs/day6_distributional_summary.csv')

=== Mortgage Balance — Computed From Project Data ===
  Source file: ons_house_prices_20260414.csv
  Latest available month: 2026-01
  UK average house price: £268,421
  LTV ratio: 75% (25% deposit)
  Estimated mortgage balance: £201,316
  Extra annual cost from +1pp shock: £2,013

  Note: this value updates automatically when main.py refreshes the UK HPI data.

=== Distributional Impact of +1pp Rate Shock ===
(Mortgage balance computed from latest UK HPI data → £2,013/year extra cost)



,decile,income_value,estimated_extra_annual_cost,extra_cost_as_pct_of_income
0,1,10725.000,2013.000,18.770
1,2,19647.000,2013.000,10.250
2,3,25009.000,2013.000,8.050
3,4,29627.000,2013.000,6.790
4,5,34244.000,2013.000,5.880
5,6,39270.000,2013.000,5.130
6,7,44887.000,2013.000,4.480
7,8,52196.000,2013.000,3.860
8,9,63025.000,2013.000,3.190
9,10,108205.000,2013.000,1.860



✓ Saved: outputs/day6_distributional_summary.csv


In [13]:
# ── Visualise ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colours = ['#d73027' if d <= 3 else '#74add1' for d in dist_df['decile']]
bars = ax.bar(dist_df['decile'], dist_df['extra_cost_as_pct_of_income'],
               color=colours, edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, dist_df['extra_cost_as_pct_of_income']):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.15,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Income Decile (1 = lowest, 10 = highest)', fontsize=11)
ax.set_ylabel('Extra Annual Cost as % of Disposable Income', fontsize=11)
ax.set_title(
    '+1pp Bank Rate Shock: Stylised Regressive Burden Across Income Deciles\n'
    f'(same £{computed_extra_cost:,.0f}/year benchmark on a £{mortgage_info["estimated_mortgage_balance"]:,.0f} mortgage)',
    fontsize=12
)
ax.set_xticks(dist_df['decile'])
ax.set_xticklabels([f'D{d}' for d in dist_df['decile']])
legend_elements = [Patch(facecolor='#d73027', label='Deciles 1–3 (most exposed)'),
                   Patch(facecolor='#74add1', label='Deciles 4–10')]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
fig.savefig(FIGURES / 'figure_02_distributional_burden.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ Saved: figures/day6/figure_02_distributional_burden.png')

✓ Saved: figures/day6/figure_02_distributional_burden.png


### Three Channels of Impact

The distributional burden of a rate rise operates through three distinct channels:

| Channel | Who is affected | Mechanism |
|---------|----------------|----------|
| **Homeowners with mortgages** | Mostly middle-to-upper deciles (D4–D8) | Direct: variable-rate mortgage payments rise immediately with Bank Rate |
| **First-time buyers** | Aspiring buyers, concentrated in D3–D6 | Higher borrowing costs reduce purchasing power; potential buyers priced out |
| **Renters and low-income households** | Disproportionately D1–D3 | Indirect: landlords face higher financing costs and pass through to rents |

### Important Nuance: The Ambiguous Net Effect

A common simplification runs as follows: *rates rise → house prices fall → first-time buyers benefit*. This framing is **incomplete and potentially misleading** for the following reasons:

1. **Credit tightening:** Higher rates tighten mortgage credit conditions. Even if nominal house prices fall, the borrowing cost may rise enough to leave monthly payments unchanged or higher.
2. **Squeezed disposable income:** For existing mortgagors, higher payments directly reduce disposable income, compressing consumption — particularly for D4–D6 households.
3. **Rental pass-through:** Landlords facing higher financing costs typically pass them on as higher rents, often with a lag of 6–18 months. Renters in D1–D3 — already spending a higher share of income on housing — absorb this second-order effect.
4. **Income share asymmetry (Day 3 finding):** Lower deciles spend approximately **27% of income on housing** versus **5% for the highest decile** (Day 1 research problem, sourced from ONS Family Spending FYE 2024). Day 3 Table 13 and Figure 6 confirmed that real income fell for all deciles during the 2022 inflation shock, with Decile 1 experiencing the steepest decline. The Day 3 pressure proxy remained exploratory and did not show a clean monotonic decile gradient, so the safest interpretation is that affordability pressure is significant but uneven across deciles. This means the same absolute cost benchmark creates a much larger proportional burden for lower-income households.

**Conclusion:** The net effect of a rate rise is ambiguous and heterogeneous. It is NOT a simple story of "rates up = prices down = good for buyers." The distributional table above quantifies one stylised dimension of this regressivity — applying the same extra mortgage-cost benchmark across deciles — but the full picture includes credit access, rent dynamics, tenure, mortgage balances, and income effects that cannot be fully captured in a single index.

---
## 05 — Policy Options Matrix

Given the distributional evidence, we compare two policy responses. The comparison is structured around objective, target group, expected benefit, main risk, fiscal cost, and economic theory basis.

In [14]:
policy_df = build_policy_matrix()
recommendation = get_primary_recommendation()

print('=== Policy Comparison Matrix ===')
display(policy_df)

policy_df.to_csv(OUTPUTS / 'day6_policy_matrix.csv', index=False)
print('\n✓ Saved: outputs/day6_policy_matrix.csv')
print(f'\n★ Primary recommendation: {recommendation["policy_name"]}')

=== Policy Comparison Matrix ===


,policy_name,objective,target_group,expected_benefit,main_risk,fiscal_cost_indication,economic_theory_basis,primary_recommendation
0,Targeted Rent Subsidy,Reduce housing cost burden for lowest-income r...,Renters in income deciles 1–3,Direct income relief; reduces effective rent-t...,May push market rents upward if housing supply...,Medium,Demand-side transfers; incidence depends on su...,True
1,Rent Controls,Cap rent growth to improve near-term affordabi...,All private renters,Immediate reduction in rental costs; broad cov...,Reduced housing supply over time as landlords ...,Low,Price ceiling set below equilibrium creates ex...,False



✓ Saved: outputs/day6_policy_matrix.csv

★ Primary recommendation: Targeted Rent Subsidy


### Policy Interpretation

**Option A — Targeted Rent Subsidy (Primary Recommendation)**

A targeted rent subsidy directed at renters in income deciles 1–3 addresses the most acutely exposed households. Economic theory predicts that a demand-side transfer's incidence depends on supply elasticity: in a low-elasticity housing market (as characterises much of the UK, especially London and urban centres), some benefit may be captured by landlords through higher equilibrium rents. This is a well-known limitation. However, it can be partly mitigated by:
- Area-based rent benchmarks as subsidy caps
- Pairing with supply-side reforms to increase rental stock

**Indicative fiscal scale:** If the subsidy covers approximately £1,500/year per household for the roughly 4 million renter households in income deciles 1–3, the annual programme cost would be in the order of £6 billion. This is comparable in scale to existing Housing Benefit expenditure but would require additional budget allocation or redistribution. The fiscal cost is real but bounded — and should be weighed against the welfare cost of inaction, which falls disproportionately on the most vulnerable households.

The targeted approach is preferred over rent controls because it preserves market signals, does not create long-run supply disincentives, and can be calibrated to the deciles where the income–housing cost gap is largest.

**Option B — Rent Controls**

Rent controls offer broad, immediate relief and require no fiscal outlay. Their theoretical and empirical drawbacks are well-established: a price ceiling set below the market-clearing rent creates excess demand, reduces incentives for new supply and maintenance investment, and can produce misallocation (e.g., low-rent tenants staying in oversized flats). The Green Party's 2026 rent-controls proposal (captured in the BBC sentiment corpus) reflects genuine public concern but faces significant supply-side objections.

**Trade-off summary:** The choice depends on the policy-maker's time horizon (short-run relief vs long-run supply health), administrative capacity (subsidy targeting vs enforcement), and tolerance for market distortions. **On balance, targeted rent subsidies are the preferred recommendation given the project's distributional evidence** — but the fiscal cost is real and requires prioritisation.

---
## 06 — Draft Conclusions

This section answers four core interpretive questions and distinguishes short-term effects, medium-term effects, and limitations.

In [15]:
# Pull numbers out of the comparison table so they can be embedded in the conclusions text below.
# Retrieve key numbers for narrative
lr_delta  = lr_row['mean_delta']
rf_delta  = rf_row['mean_delta']
lr_pct    = lr_row['pct_change']
rf_pct    = rf_row['pct_change']

lr_mae    = metrics_df[metrics_df['model']=='linear_regression']['MAE'].values[0]
lr_r2     = metrics_df[metrics_df['model']=='linear_regression']['R2'].values[0]
rf_mae    = metrics_df[metrics_df['model']=='random_forest']['MAE'].values[0]
rf_r2     = metrics_df[metrics_df['model']=='random_forest']['R2'].values[0]

d1_pct    = dist_df[dist_df['decile']==1]['extra_cost_as_pct_of_income'].values[0]
d10_pct   = dist_df[dist_df['decile']==10]['extra_cost_as_pct_of_income'].values[0]

print(f'LR scenario delta : {lr_delta:.2f} index points ({lr_pct:.2f}%)')
print(f'RF scenario delta : {rf_delta:.2f} index points ({rf_pct:.2f}%)')
print(f'LR holdout R²     : {lr_r2:.3f}  MAE: {lr_mae:.3f}')
print(f'RF holdout R²     : {rf_r2:.3f}  MAE: {rf_mae:.3f}')
print(f'D1 extra cost share: {d1_pct:.1f}%  vs D10: {d10_pct:.1f}%')

LR scenario delta : 1.07 index points (1.10%)
RF scenario delta : 1.14 index points (1.16%)
LR holdout R²     : -28.506  MAE: 6.377
RF holdout R²     : -7.386  MAE: 3.371
D1 extra cost share: 18.8%  vs D10: 1.9%


In [16]:
# ── Write conclusions to file ─────────────────────────────────────────────────
conclusions_text = f"""# Day 6 Conclusions — Policy Interpretation and Findings

**Project:** The Impact of Inflation and Housing Costs on Social Inequality  
**Date generated:** 2026-04-21  
**Data window:** June 2021 – January 2026 (56 months)  

---

## Executive Summary

This project finds consistent evidence that inflation and rising housing costs impose a disproportionately large burden on lower-income households in the UK. Real-income erosion was concentrated in the lowest deciles during the 2021–2024 high-inflation period (H1 supported). Housing costs as a share of income are highest for the lowest deciles and lowest for the highest decile (H2 supported). The Day 4 models demonstrate that Bank Rate and inflation have a detectable association with house price movements, though holdout R² is negative for both models, limiting their use as forecasters (H3 partially supported). A +1 percentage point Bank Rate scenario produces a small positive model-implied HPI delta of about {lr_delta:.1f} index points under the linear model and {rf_delta:.1f} index points under the random forest. This is not a causal estimate or proof that higher rates raise house prices; it reflects the fitted lag structure in a short macro window. The same £{computed_extra_cost:,.0f} extra annual mortgage-cost benchmark is applied across deciles and represents approximately {d1_pct:.0f}% of Decile 1 income but only {d10_pct:.1f}% of Decile 10 income, illustrating the regressive structure. A targeted rent subsidy for renters in deciles 1–3 is the preferred policy response over rent controls, as it directly addresses the income–housing cost gap while avoiding long-run supply distortions.

---

## Full Conclusions

### 1. What does the model suggest?

The Day 4 models (linear regression and random forest) were trained on 56 months of UK macro data to predict the House Price Index from lagged Bank Rate, inflation, and unemployment. The +1pp rate shock scenario yields:

- **Linear regression:** mean delta = {lr_delta:.2f} index points ({lr_pct:.2f}% relative change)
- **Random forest:** mean delta = {rf_delta:.2f} index points ({rf_pct:.2f}% relative change)

The plotted delta is **scenario minus baseline**, so the positive values mean the fitted models predict slightly higher HPI under the shocked lag values. This should be read cautiously: it does **not** mean Bank Rate causally increases house prices. It means that, in this short 2021–2026 window and this specific lag structure, the shocked Bank Rate features are associated with a small positive fitted HPI change. This can coexist with the broader economic expectation that higher rates often cool housing demand, because the model is capturing correlations from one unusual macro period rather than an identified causal mechanism.

The linear-regression line is nearly flat because the same +1pp shock is applied to the same two lagged Bank Rate features in every row, and the model applies one fixed coefficient structure. The random forest varies more over time because it can respond nonlinearly to the surrounding monthly conditions. The holdout evaluation (Day 5) reveals that both models produce **negative R²** ({lr_r2:.3f} for LR, {rf_r2:.3f} for RF), meaning neither outperforms a naïve mean prediction on out-of-sample data. The models are best understood as **descriptive association tools**, not reliable forecasters.

### 2. Who is most affected?

The distributional evidence points clearly to lower-income households as the most exposed group:

- **Real income erosion (H1):** Day 3 analysis found that real income fell most sharply for Decile 1 during the 2022–2023 inflation surge. The CPIH deflator applied uniformly, but lower-income households spend a higher share on necessities (food, energy, housing) — the categories that drove CPI above headline levels.
- **Housing cost share (H2):** Lower deciles spend approximately 27% of income on housing vs 5% for the highest decile. Any absolute rise in housing costs (mortgage payments, rents) therefore represents a larger proportional burden.
- **Distributional impact matrix:** A stylised £{computed_extra_cost:,.0f}/year extra-cost benchmark from a +1pp rate shock represents {d1_pct:.1f}% of Decile 1 income but only {d10_pct:.1f}% of Decile 10 income — a ratio of roughly {d1_pct/d10_pct:.0f}:1. The same benchmark is applied to all deciles to illustrate regressivity; it is not a claim that every household faces exactly this payment increase.

Three channels operate simultaneously:
1. Mortgagors (mostly D4–D8) face direct payment increases
2. First-time buyers (D3–D6) face tighter credit and higher monthly costs
3. Renters and low-income households (D1–D3) face indirect rent pass-through from landlords

### 3. Which policy makes most sense?

**Primary recommendation: Targeted rent subsidy for renters in income deciles 1–3.**

Rationale:
- Directly addresses the income–housing cost gap for the most exposed households
- Preserves market price signals (unlike rent controls)
- Avoids long-run supply-side disincentives
- Can be calibrated and means-tested

Trade-offs:
- Fiscal cost is real (medium-level public expenditure)
- Incidence: if housing supply is inelastic, some subsidy may be captured by landlords through higher rents — mitigated by area-based rent benchmarks
- Does not directly address underlying supply shortage

Rent controls offer immediate, broad relief at low fiscal cost, but the economic consensus and historical evidence (UK 1970s, Sweden, New York) suggest they reduce private rental supply over the medium term and create misallocation. For a project explicitly focused on improving outcomes for lower-income deciles over the long run, subsidies are the more structurally sound instrument.

### 4. What are the limits?

**Model limitations:**
- Negative holdout R² confirms neither model is a reliable forecaster outside the training period
- 56-month window covers a single rate-hiking cycle — the models cannot separate Bank Rate from simultaneous post-pandemic demand and energy price shocks
- No rent-specific price index: the House Price Index used here reflects property values, not rental costs
- Lagged features (1-month, 3-month) may not capture the full transmission lag of monetary policy (typically 12–18 months)

**Data limitations:**
- Annual income data (decile data available only for 5 years) limits the granularity of real-income analysis
- CPIH applied as a uniform deflator across deciles, despite differential expenditure weights
- BBC sentiment corpus (50 headlines) is too small and too recent to constitute a reliable longitudinal measure; it provides media context, not economic measurement

**Inference limitations:**
- The scenario analysis is a counterfactual, not a forecast
- Correlations established in the training period may not persist under different macro regimes
- No causal identification strategy — results cannot rule out omitted variable bias

---

## Short-term, Medium-term, and Long-term Effects

### Short-term effects (0–6 months after a rate rise)
- Variable-rate mortgage payments increase immediately; households with tracker mortgages face direct income squeeze
- Fixed-rate borrowers are temporarily insulated but face reset risk at refinancing
- Borrowing demand may soften, although the model-implied HPI scenario should not be treated as a reliable house-price forecast
- Consumer confidence typically dips (corroborated by the predominantly negative BBC sentiment in the corpus)

### Medium-term effects (6–24 months)
- If prices cool, first-time buyer affordability *may* improve on the price dimension — but higher borrowing costs can fully offset this, leaving monthly mortgage costs unchanged or higher
- Rental pass-through: landlords facing higher financing costs typically raise rents with a 6–18 month lag; D1–D3 renters absorb this second-order effect
- Credit tightening reduces the pool of qualifying buyers, further dampening transaction volumes
- Net effect is ambiguous and heterogeneous: it is not a simple "rates up = prices down = good for buyers" story

### Long-term (beyond 24 months)
- Supply response: if developer activity contracts in response to higher financing and lower demand, the long-run stock of housing could fall, worsening affordability at lower price points
- Income inequality dynamics: sustained higher rates that compress real wages (via reduced business investment) could widen the income gap, making the distributional burden worse over time
- These long-run dynamics are beyond the scope of the current 56-month model window

---

## 5 Actionable Insights for the Day 7 Report

1. **Lead with the regressivity finding:** The £{computed_extra_cost:,.0f} extra cost absorbs {d1_pct:.0f}% of D1 income vs {d10_pct:.1f}% of D10 income. This is the project's most concrete quantitative contribution.
2. **Qualify the model results:** Present the scenario delta alongside the negative R² to make clear that the models are association tools, not forecasters. Avoid presenting the scenario as a prediction.
3. **Distinguish the three impact channels** (mortgagors, first-time buyers, renters) in the report narrative — they are affected by different mechanisms and on different timescales.
4. **Recommend targeted rent subsidy with caveats:** State the incidence risk explicitly and link it to the need for complementary supply-side reform.
5. **Flag the data gaps as a research agenda:** Rent-specific index, regional breakdowns, and a longer time series would materially strengthen the analysis. Frame limitations as opportunities for future work.
"""

with open(OUTPUTS / 'day6_conclusions.md', 'w', encoding='utf-8') as f:
    f.write(conclusions_text)

print('✓ Saved: outputs/day6_conclusions.md')
print(f'  Length: {len(conclusions_text)} characters')

✓ Saved: outputs/day6_conclusions.md
  Length: 9388 characters


---
## 07 — Day 7 Asset Pack

Listing all figures, tables, and key messages to be assembled into the Day 7 final report.

In [17]:
print('=== Day 7 Report Asset Pack ===')
print()
print('── FIGURES (include in report) ──────────────────────────────────────────')
print()

figure_registry = [
    # Day 3
    ('figures/day3/figure_01_macro_distributions.png',
     'Macro variable distributions (inflation, HPI, unemployment, bank rate)'),
    ('figures/day3/figure_02_income_by_decile_bar.png',
     'Income by decile across all years — shows widening gap'),
    ('figures/day3/figure_03_income_vs_housing.png',
     'Income vs housing cost scatter — illustrates regressive structure'),
    ('figures/day3/figure_05_correlation_heatmap_pearson_spearman.png',
     'Correlation heatmap — Bank Rate / inflation vs HPI'),
    ('figures/day3/figure_06_social_pressure_real_income_index.png',
     'Real income index by decile — visual evidence for H1'),
    ('figures/day3/figure_07_social_pressure_housing_vs_income.png',
     'Housing cost as share of income by decile — evidence for H2'),
    ('figures/day3/figure_08_decile_pressure_proxy_bar.png',
     'Pressure proxy bar chart — D1–D3 most exposed'),
    # Day 5
    ('figures/day5/figure_01_closed_holdout_actual_vs_predicted.png',
     'Holdout actual vs predicted — both models, showing model limitations'),
    ('figures/day5/figure_02_closed_holdout_residuals_over_time.png',
     'Residuals over time — shows systematic under-prediction'),
    ('figures/day5/day5_sentiment.png',
     'BBC sentiment distribution — media context'),
    # Day 6
    ('figures/day6/figure_01_rate_scenario_delta.png',
     '+1pp rate scenario delta — both models over time'),
    ('figures/day6/figure_02_distributional_burden.png',
     'Regressive burden chart — extra cost as % income by decile'),
]

for path, description in figure_registry:
    exists = (PROJECT_ROOT / path).exists()
    status = '✓' if exists else '⚠'
    print(f'  {status}  {path}')
    print(f'      → {description}')

print()
print('── DATA TABLES (include in report appendix) ────────────────────────────')
tables = [
    ('outputs/day5_official_holdout_metrics.csv', 'Model holdout metrics (MAE, RMSE, R²)'),
    ('outputs/day6_interest_scenario.csv', '+1pp scenario comparison table'),
    ('outputs/day6_distributional_summary.csv', 'Decile distributional impact matrix'),
    ('outputs/day6_policy_matrix.csv', 'Policy comparison matrix'),
]
for path, description in tables:
    print(f'  • {path} — {description}')

=== Day 7 Report Asset Pack ===

── FIGURES (include in report) ──────────────────────────────────────────

  ✓  figures/day3/figure_01_macro_distributions.png
      → Macro variable distributions (inflation, HPI, unemployment, bank rate)
  ✓  figures/day3/figure_02_income_by_decile_bar.png
      → Income by decile across all years — shows widening gap
  ✓  figures/day3/figure_03_income_vs_housing.png
      → Income vs housing cost scatter — illustrates regressive structure
  ✓  figures/day3/figure_05_correlation_heatmap_pearson_spearman.png
      → Correlation heatmap — Bank Rate / inflation vs HPI
  ✓  figures/day3/figure_06_social_pressure_real_income_index.png
      → Real income index by decile — visual evidence for H1
  ✓  figures/day3/figure_07_social_pressure_housing_vs_income.png
      → Housing cost as share of income by decile — evidence for H2
  ✓  figures/day3/figure_08_decile_pressure_proxy_bar.png
      → Pressure proxy bar chart — D1–D3 most exposed
  ✓  figures/day5/fi

### Repository and optional dashboard

**GitHub repository structure:** The full project repository should include:
- `notebooks/` — Day 2 pipeline notebook, Day 3 EDA, Day 5 evaluation, Day 6 interpretation
- `src/` — reusable Python modules (`main.py`, `day6_interest_scenario.py`, `day6_policy_analysis.py`)
- `data/raw/` — original downloaded files with collection dates in filenames
- `data/processed/` — cleaned CSVs produced by the pipeline
- `outputs/` — metrics, scenario tables, conclusions, key messages
- `figures/` — all saved PNGs organised by day (day3/, day5/, day6/)
- `docs/` — data log, data dictionary

**Streamlit dashboard (if time permits on Day 7):** A simple interactive app with three panels:
1. Macro time series viewer (inflation, HPI, Bank Rate, unemployment from Day 3)
2. Distributional burden bar chart (extra cost as % of income by decile from Day 6)
3. Scenario comparison table (baseline vs +1pp predictions from Day 6)

This is optional and depends on Day 7 time allocation.

In [18]:
# ── Key messages file ─────────────────────────────────────────────────────────
key_messages = f"""Day 6 Key Messages — UK Inflation, Housing Costs, and Social Inequality
Generated: 2026-04-21
===========================================================================

• REGRESSIVITY OF HOUSING COSTS: The same stylised £{computed_extra_cost:,.0f}/year mortgage-cost benchmark
  absorbs ~{d1_pct:.0f}% of Decile 1 income but only ~{d10_pct:.1f}% of Decile 10 income — a ~{d1_pct/d10_pct:.0f}:1 ratio,
  illustrating the regressive structure of housing cost shocks (H2 supported).

• REAL INCOME EROSION (H1 SUPPORTED): Inflation in 2022–2024 disproportionately eroded
  real incomes of lower deciles, who spend a higher share on necessities. Day 3 analysis
  confirms real income fell most for Decile 1 during the peak inflation period.

• MODEL-IMPLIED RATE EFFECT (H3 PARTIAL): A +1pp Bank Rate scenario produces an
  estimated +{lr_delta:.1f} index-point HPI delta (linear model) and +{rf_delta:.1f} points (random forest).
  Negative holdout R² ({lr_r2:.2f} LR, {rf_r2:.2f} RF) limits confidence in these as forecasts;
  treat as descriptive associations within the June 2021–January 2026 model-ready window only, not causal effects.

• AMBIGUOUS NET EFFECT FOR BUYERS: A possible rate-driven house price slowdown does NOT straightforwardly
  benefit first-time buyers — higher borrowing costs, tighter credit, and rental pass-through
  can offset or reverse any price-level benefit. The net effect is heterogeneous across groups.

• MEDIA SENTIMENT CORROBORATES CONCERN: BBC headline sentiment corpus (50 headlines, April 2026)
  is predominantly negative/neutral on housing and inflation topics, consistent with public concern
  about affordability. Rent controls feature prominently, reflecting political pressure on housing.

• POLICY RECOMMENDATION — TARGETED RENT SUBSIDY: Preferred over rent controls because it
  addresses the income-housing gap directly, preserves supply incentives, and can be targeted
  to the most exposed deciles. Fiscal cost is medium but manageable; supply-side reform needed
  in parallel to avoid landlord incidence.

• DATA GAPS AS RESEARCH AGENDA: Key missing data — rent-specific price index, regional
  breakdowns, household-level micro data, richer lag structures — represent clear next steps
  to strengthen the evidence base for policy recommendations.
"""

with open(OUTPUTS / 'day6_key_messages.txt', 'w', encoding='utf-8') as f:
    f.write(key_messages)

print('✓ Saved: outputs/day6_key_messages.txt')
print()
print(key_messages)

✓ Saved: outputs/day6_key_messages.txt

Day 6 Key Messages — UK Inflation, Housing Costs, and Social Inequality
Generated: 2026-04-21

• REGRESSIVITY OF HOUSING COSTS: The same stylised £2,013/year mortgage-cost benchmark
  absorbs ~19% of Decile 1 income but only ~1.9% of Decile 10 income — a ~10:1 ratio,
  illustrating the regressive structure of housing cost shocks (H2 supported).

• REAL INCOME EROSION (H1 SUPPORTED): Inflation in 2022–2024 disproportionately eroded
  real incomes of lower deciles, who spend a higher share on necessities. Day 3 analysis
  confirms real income fell most for Decile 1 during the peak inflation period.

• MODEL-IMPLIED RATE EFFECT (H3 PARTIAL): A +1pp Bank Rate scenario produces an
  estimated +1.1 index-point HPI delta (linear model) and +1.1 points (random forest).
  Negative holdout R² (-28.51 LR, -7.39 RF) limits confidence in these as forecasts;
  treat as descriptive associations within the June 2021–January 2026 model-ready window only, not caus

---
## 08 — Limitations and Next Steps

This section catalogues what the project can and cannot claim, and bridges to the Day 7 final report.

### Short-term limitations (inherent to this dataset and window)

1. **Small sample — 56 months:** The model-ready dataset covers June 2021 to January 2026, a period dominated by a single macro event (post-pandemic inflation + Bank Rate hiking cycle). The models' parameters are heavily shaped by this specific episode and may not generalise.

2. **No rent-specific index:** The dependent variable is the ONS House Price Index (property values), not a rental price index. The analysis therefore captures one dimension of housing costs but not the rental market, which is the primary market for lower-income households.

3. **CPIH as uniform deflator:** Income deflation in the decile analysis uses CPIH, which is a national average. Lower-income households have different expenditure weights — higher shares on food and energy — so CPIH likely *understates* real income erosion for Decile 1 relative to Decile 10.

4. **Annual income data:** ONS income by decile is available only annually (5 data points: 2020–2024). This limits the precision of the real-income analysis and prevents month-level distributional tracking.

5. **Short lag structure:** The model uses 1-month and 3-month lags for Bank Rate. The monetary policy transmission to house prices typically operates over a 12–18 month horizon. The models may therefore be capturing only the initial, partial effect of rate changes.

### Medium-term improvements

1. **Regional breakdowns:** Housing cost pressure is highly heterogeneous across UK regions (London vs North England). Regional analysis would reveal spatial inequality not visible in national aggregates.

2. **Rental price index:** Incorporating the ONS Private Rental Index would allow direct analysis of rental market dynamics, which are more relevant for lower-income households than the property transaction index.

3. **Richer lag structures (VAR/VECM):** A Vector Autoregression or cointegration framework would capture dynamic interactions between Bank Rate, inflation, unemployment, and house prices more rigorously, and would allow impulse response analysis as a more principled alternative to the counterfactual shock.

4. **Larger and longer BBC corpus:** 50 headlines from a single retrieval date do not constitute a time series. A properly constructed longitudinal sentiment dataset (monthly scores over 2–5 years) would allow sentiment to be used as an independent feature rather than descriptive context.

5. **Household-level micro data:** Using the ONS Living Costs and Food Survey or Family Resources Survey would allow mortgage/rent/income analysis at the household level, removing the need for decile-level approximations.

### What this project can and cannot claim

| Can claim | Cannot claim |
|-----------|-------------|
| Inflation and housing costs impose a **regressive burden** on lower-income deciles (supported by Day 3 and Day 6 distributional analysis) | That Bank Rate *causes* house price movements (no causal identification) |
| Bank Rate, inflation, and unemployment are **associated** with HPI in the 2021–2026 window | That the model-implied scenario delta is a **reliable forecast** |
| Lower deciles experienced greater **real income erosion** during the 2022–2024 inflation peak | That the effect holds outside the observed macro regime |
| BBC sentiment reflects **public concern** about housing affordability | That sentiment caused or predicted economic outcomes |
| Targeted rent subsidies have **better theoretical properties** than rent controls for long-run supply health | That any specific policy will achieve its stated objective without unintended consequences |

### Bridge to Day 7

The interpretive material in this notebook — the scenario analysis, distributional mapping, policy comparison, and draft conclusions — is ready to be assembled into the final report. Day 7 should:

1. Synthesise the Day 3 descriptive evidence, Day 4–5 modelling results, and Day 6 policy interpretation into a coherent narrative
2. Lead with the executive summary from `outputs/day6_conclusions.md`
3. Use the key messages from `outputs/day6_key_messages.txt` as section headers or call-out boxes
4. Include figures as listed in the Day 7 asset pack (Section 07 above)
5. Close with the limitations section and future research agenda from Section 08

**All Day 6 outputs are in `outputs/` and `figures/day6/`. The final report can be assembled from these files without rerunning any modelling code.**

In [19]:
# ── Final deliverables check ───────────────────────────────────────────────────
print('=== Day 6 Deliverables Checklist ===')
deliverables = [
    ('outputs/day6_interest_scenario.csv',       'Scenario comparison table'),
    ('outputs/day6_distributional_summary.csv',  'Decile distributional impact'),
    ('outputs/day6_policy_matrix.csv',           'Policy comparison matrix'),
    ('outputs/day6_conclusions.md',              'Executive summary + full conclusions'),
    ('outputs/day6_key_messages.txt',            'Key messages (5–7 bullets)'),
    ('figures/day6/figure_01_rate_scenario_delta.png',    'Scenario delta figure'),
    ('figures/day6/figure_02_distributional_burden.png',  'Distributional burden figure'),
    ('src/day6_interest_scenario.py',            'Reusable scenario module'),
    ('src/day6_policy_analysis.py',              'Reusable policy analysis module'),
]
all_ok = True
for rel_path, description in deliverables:
    full_path = PROJECT_ROOT / rel_path
    exists = full_path.exists()
    status = '✓' if exists else '✗ MISSING'
    if not exists:
        all_ok = False
    print(f'  {status}  {rel_path:55s}  {description}')

print()
if all_ok:
    print('All Day 6 deliverables present. Ready for Day 7 assembly.')
else:
    print('Some deliverables are missing — check cells above for errors.')

=== Day 6 Deliverables Checklist ===
  ✓  outputs/day6_interest_scenario.csv                       Scenario comparison table
  ✓  outputs/day6_distributional_summary.csv                  Decile distributional impact
  ✓  outputs/day6_policy_matrix.csv                           Policy comparison matrix
  ✓  outputs/day6_conclusions.md                              Executive summary + full conclusions
  ✓  outputs/day6_key_messages.txt                            Key messages (5–7 bullets)
  ✓  figures/day6/figure_01_rate_scenario_delta.png           Scenario delta figure
  ✓  figures/day6/figure_02_distributional_burden.png         Distributional burden figure
  ✓  src/day6_interest_scenario.py                            Reusable scenario module
  ✓  src/day6_policy_analysis.py                              Reusable policy analysis module

All Day 6 deliverables present. Ready for Day 7 assembly.
